In [0]:
flights_data = [
("F101","Indigo","Hyderabad","Delhi",140,"On Time"),
("F102","Air India","Mumbai","Chennai",120,"Delayed"),
("F103","Vistara","Bangalore","Hyderabad",90,"On Time"),
("F104","Indigo","Delhi","Mumbai",130,"Cancelled"),
("F105","Air India","Chennai","Bangalore",80,"On Time"),
("F106","Akasa","Pune","Delhi",150,"Delayed"),
("F107","Vistara","Hyderabad","Kolkata",160,"On Time"),
("F108","Indigo","Mumbai","Hyderabad",110,"On Time"),
("F109","Akasa","Delhi","Chennai",145,"Delayed"),
("F110","Air India","Bangalore","Mumbai",95,"On Time"),
("F111","Indigo","Hyderabad","Goa",75,"On Time"),
("F112","Vistara","Goa","Delhi",150,"Cancelled"),
("F113","Akasa","Chennai","Pune",100,"On Time"),
("F114","Air India","Kolkata","Bangalore",170,"Delayed"),
("F115","Indigo","Delhi","Hyderabad",135,"On Time")
]

columns = [
    "flight_id",
    "airline",
    "from_city",
    "to_city",
    "duration",
    "status"
]

df_flights = spark.createDataFrame(flights_data, columns)

display(df_flights)

flight_id,airline,from_city,to_city,duration,status
F101,Indigo,Hyderabad,Delhi,140,On Time
F102,Air India,Mumbai,Chennai,120,Delayed
F103,Vistara,Bangalore,Hyderabad,90,On Time
F104,Indigo,Delhi,Mumbai,130,Cancelled
F105,Air India,Chennai,Bangalore,80,On Time
F106,Akasa,Pune,Delhi,150,Delayed
F107,Vistara,Hyderabad,Kolkata,160,On Time
F108,Indigo,Mumbai,Hyderabad,110,On Time
F109,Akasa,Delhi,Chennai,145,Delayed
F110,Air India,Bangalore,Mumbai,95,On Time


In [0]:
# ==========================================
# Create Booking DataFrame
# ==========================================

bookings_data = [
("B1001","F101","Rahul Sharma","Economy",8500,"2026-06-01"),
("B1002","F101","Priya Reddy","Business",22000,"2026-06-01"),
("B1003","F102","Amit Kumar","Economy",9000,"2026-06-02"),
("B1004","F103","Sneha Patel","Premium Economy",15000,"2026-06-02"),
("B1005","F104","Farhan Ali","Economy",7500,"2026-06-03"),
("B1006","F105","Neha Singh","Business",25000,"2026-06-03"),
("B1007","F106","Arjun Verma","Economy",10000,"2026-06-04"),
("B1008","F107","Meera Nair","Premium Economy",17000,"2026-06-04"),
("B1009","F108","Kiran Rao","Economy",9500,"2026-06-05"),
("B1010","F109","Nisha Reddy","Business",28000,"2026-06-05"),
("B1011","F110","David Thomas","Economy",8000,"2026-06-06"),
("B1012","F111","Ayesha Khan","Premium Economy",16000,"2026-06-06"),
("B1013","F112","Rohit Sharma","Economy",7000,"2026-06-07"),
("B1014","F113","Pooja Mehta","Business",24000,"2026-06-07"),
("B1015","F114","Sanjay Gupta","Economy",10500,"2026-06-08"),
("B1016","F115","Divya Iyer","Premium Economy",18000,"2026-06-08"),
("B1017","F101","Vikram Singh","Economy",8500,"2026-06-09"),
("B1018","F103","Anjali Rao","Business",23000,"2026-06-09"),
("B1019","F107","Faiz Ahmed","Economy",9500,"2026-06-10"),
("B1020","F110","Megha Kapoor","Premium Economy",15500,"2026-06-10")
]

columns = [
    "booking_id",
    "flight_id",
    "passenger_name",
    "travel_class",
    "ticket_price",
    "booking_date"
]

df_bookings = spark.createDataFrame(
    bookings_data,
    columns
)

In [0]:
df_flights.createOrReplaceTempView("flights")

df_bookings.createOrReplaceTempView("bookings")

print("Temp Views Created Successfully")

Temp Views Created Successfully


In [0]:
revenue_airline = spark.sql("""

SELECT
    f.airline,
    SUM(b.ticket_price) AS total_revenue

FROM bookings b

JOIN flights f
ON b.flight_id = f.flight_id

GROUP BY f.airline

ORDER BY total_revenue DESC

""")

display(revenue_airline)

airline,total_revenue
Indigo,90000
Vistara,71500
Air India,68000
Akasa,62000


In [0]:
revenue_route = spark.sql("""

SELECT
    CONCAT(f.from_city,' -> ',f.to_city) AS route,
    SUM(b.ticket_price) AS total_revenue

FROM bookings b

JOIN flights f
ON b.flight_id = f.flight_id

GROUP BY
    f.from_city,
    f.to_city

ORDER BY total_revenue DESC

""")

display(revenue_route)

route,total_revenue
Hyderabad -> Delhi,39000
Bangalore -> Hyderabad,38000
Delhi -> Chennai,28000
Hyderabad -> Kolkata,26500
Chennai -> Bangalore,25000
Chennai -> Pune,24000
Bangalore -> Mumbai,23500
Delhi -> Hyderabad,18000
Hyderabad -> Goa,16000
Kolkata -> Bangalore,10500


In [0]:
avg_price = spark.sql("""

SELECT
    ROUND(AVG(ticket_price),2)
    AS average_ticket_price

FROM bookings

""")

display(avg_price)

average_ticket_price
14575.0


In [0]:
popular_destination = spark.sql("""

SELECT
    f.to_city,
    COUNT(*) AS total_bookings

FROM bookings b

JOIN flights f
ON b.flight_id = f.flight_id

GROUP BY f.to_city

ORDER BY total_bookings DESC

LIMIT 1

""")

display(popular_destination)

to_city,total_bookings
Delhi,5
